# Tableau de bord énergétique

Ce tableau de bord permet de mieux comprendre la performance énergétique des logements :

- Quelle est la répartition des logements selon leur classe DPE ?
- Est-ce que les logements anciens consomment plus ?
- Quelles énergies sont les plus polluantes ?


In [1]:
import pandas as pd
import plotly.express as px

In [2]:
#data =  pd.read_csv('C:\\Users\\HP\\Desktop\\Master2\\ML\\m2_enedis_dpe_app-main\\data\\df_adem_enedis_iris_69_prepared.csv')

In [3]:
#data =  data.head(1000)
import os

import requests


API_URL = os.getenv("API_URL", "https://riadshrn-api-dpe-conso.hf.space/data/visualisation?page=1&size=10000")
response = requests.get(API_URL)
data = response.json()
df = pd.DataFrame(data["data"])   

In [4]:
data =  df

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 24 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   emission_ges_5_usages              10000 non-null  float64
 1   nom_commune_ban                    10000 non-null  object 
 2   type_energie_principale_chauffage  10000 non-null  object 
 3   qualite_isolation_murs             10000 non-null  object 
 4   type_batiment                      10000 non-null  object 
 5   conso_ecs_ep                       10000 non-null  float64
 6   surface_habitable_logement         10000 non-null  float64
 7   conso_chauffage_ep                 10000 non-null  float64
 8   isolation_toiture                  10000 non-null  float64
 9   etiquette_dpe                      10000 non-null  object 
 10  code_postal_ban                    10000 non-null  int64  
 11  zone_climatique                    10000 non-null  obje

In [6]:
total = len(data)
part_f_g = (data['etiquette_dpe'].isin(['F','G']).mean() * 100)
conso_moy = data['conso_totale_mwh'].mean()
conso_m2_moy = data['conso_m2'].mean()

print(f" Nombre de logements : {total:,}".replace(',', ' '))
print(f" Part des logements énergivores (F/G) : {part_f_g:.1f} %")
print(f" Conso totale moyenne : {conso_moy:.2f} MWh / logement")
print(f" Conso moyenne par m² : {conso_m2_moy:.1f} kWh/m²")


 Nombre de logements : 10 000
 Part des logements énergivores (F/G) : 2.6 %
 Conso totale moyenne : 16326.02 MWh / logement
 Conso moyenne par m² : 152.8 kWh/m²


In [12]:
# MAJ (facultatif mais conseillé)
!pip install -U plotly

import plotly.express as px
import plotly.io as pio

# Meilleur choix sous VS Code Notebook
pio.renderers.default = "notebook_connected"   # 1er essai
# pio.renderers.default = "iframe_connected"   # si le précédent n'affiche rien
# pio.renderers.default = "browser"            # ouvre dans le navigateur si besoin


  Using cached plotly-6.3.1-py3-none-any.whl.metadata (8.5 kB)
Using cached plotly-6.3.1-py3-none-any.whl (9.8 MB)
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1


In [15]:
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"

fig = px.pie(
    data,
    names='etiquette_dpe',
    color='color_dpe',
    title='🏡 Répartition des logements selon la classe DPE',
    hole=0.3
)
fig.update_traces(textinfo='percent+label', textfont_size=14)
fig.update_layout(showlegend=False)
fig.show()


In [8]:
moy_dpe = data.groupby(['etiquette_dpe', 'color_dpe'], as_index=False)['conso_totale_mwh'].mean()

fig = px.bar(
    moy_dpe,
    x='etiquette_dpe',
    y='conso_totale_mwh',
    color='color_dpe',
    text_auto='.1f',
    title='💡 Consommation moyenne d’énergie selon la classe DPE'
)
fig.update_layout(showlegend=False, plot_bgcolor='white')
fig.show()


In [9]:
moy_anc = data.groupby('classe_annee_construction', as_index=False)['conso_totale_mwh'].mean()

fig = px.line(
    moy_anc,
    x='classe_annee_construction',
    y='conso_totale_mwh',
    markers=True,
    title='🏠 Consommation moyenne d’énergie selon la période de construction'
)
fig.update_traces(line_color='#FF7F0E', marker=dict(size=10))
fig.update_layout(plot_bgcolor='white')
fig.show()


In [10]:
fig = px.box(
    data,
    x='type_energie_principale_chauffage',
    y='emission_ges_5_usages',
    color='type_energie_principale_chauffage',
    title='🌍 Émissions de gaz à effet de serre selon le type d’énergie principale'
)
fig.update_layout(showlegend=False, plot_bgcolor='white')
fig.show()


In [11]:
import plotly.express as px

tmp = data.assign(
    groupe_dpe = data['etiquette_dpe'].map(lambda x: 'Performants (A–D)' if x in ['A','B','C','D'] else 'Énergivores (E–G)')
)

fig = px.pie(
    tmp,
    names='groupe_dpe',
    title='🏷️ Part des logements performants vs énergivores',
    hole=0.45,
    color='groupe_dpe',
    color_discrete_map={'Performants (A–D)':'#2ca02c','Énergivores (E–G)':'#d62728'}
)
fig.update_traces(textinfo='percent+label')
fig.update_layout(showlegend=False)
fig.show()
